In [0]:
df = spark.read.table("spotifyde_catalog.silver.user")
display(df)

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-8644278593863610>, line 2
      1 df = spark.read.table("spotifyde_catalog.silver.user")
----> 2 display(df)

File <command-8644278593863610>, line 1
----> 1 df = spark.read.table("spotifyde_catalog.silver.user")
      2 display(df)

AnalysisException: [TABLE_OR_VIEW_NOT_FOUND] The table or view `spotifyde_catalog`.`silver`.`user` cannot be found. Verify the spelling and correctness of the schema and catalog.
Search path: [`system`.`session`, `system`.`builtin`, `system`.`ai`, `adbspotifyde`.`default`].
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS. SQLSTATE: 42P01;
'UnresolvedRelation [spotifyde_catalog, silver, user], [], false


JVM stacktrace:
org.ap

In [0]:
%sql
--Verifying Silver table and quality check (as shouldn't have duplicates)

SELECT * 
FROM 
spotifyde_catalog.silver.user


user_id,user_name,country,subscription_type,start_date,end_date,updated_at,first_name,last_name
69,Gregory Hart,Syrian Arab Republic,Premium,2025-01-10,null,2025-10-05T19:49:55.000Z,Gregory,Hart
129,Stephanie Robinson,Malaysia,Family,2025-04-22,null,2025-10-06T19:49:55.000Z,Stephanie,Robinson
166,Daniel Rodriguez,Maldives,Family,2023-10-27,null,2025-09-16T19:49:55.000Z,Daniel,Rodriguez
448,Deanna Mckinney,Vanuatu,Family,2024-05-13,null,2025-09-20T19:49:55.000Z,Deanna,Mckinney
467,Emily Franklin,British Indian Ocean Territory (Chagos Archipelago),Premium,2025-05-25,null,2025-09-09T19:49:55.000Z,Emily,Franklin
83,Sarah Hinton,Christmas Island,Free,2023-11-21,null,2025-09-18T19:49:55.000Z,Sarah,Hinton
110,Richard Martinez,Isle of Man,Free,2023-12-19,null,2025-10-01T19:49:55.000Z,Richard,Martinez
150,Abigail Olsen,Uruguay,Free,2024-03-04,null,2025-10-07T19:49:55.000Z,Abigail,Olsen
369,Casey Carpenter,Heard Island and McDonald Islands,Family,2024-07-05,null,2025-10-05T19:49:55.000Z,Casey,Carpenter
413,Jennifer Payne,Antarctica (the territory South of 60 deg S),Premium,2025-05-03,null,2025-09-13T19:49:55.000Z,Jennifer,Payne


In [0]:
%sql
--Check whether user_id is unique:

SELECT 
 user_id,
 count(*) as cnt
FROM spotifyde_catalog.silver.user
GROUP BY user_id
HAVING COUNT(user_id) > 1

--Ideally this returns zero rows.
--Because Silver represents the current cleaned entity.

user_id,cnt


## Creating the initial Gold dimension

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS spotifyde_catalog.gold

In [0]:
%sql

-- ============================================================
-- STEP 1: INITIAL GOLD TABLE
-- Run this ONLY ONCE
-- ============================================================
-- Silver is the source.
-- Gold gets a surrogate key (DimUserKey).
-- ROW_NUMBER() creates the initial surrogate keys.

CREATE TABLE IF NOT EXISTS spotifyde_catalog.gold.dim_user
USING DELTA
LOCATION 'abfss://gold@adlspotifyde.dfs.core.windows.net/dim_user/'
AS
SELECT
    ROW_NUMBER() OVER (ORDER BY user_id) AS DimUserKey,  -- Surrogate key
    user_id,
    user_name,
    country,
    subscription_type,
    start_date,
    end_date,
    updated_at
FROM spotifyde_catalog.silver.user;

num_affected_rows,num_inserted_rows


In [0]:
""" Pyspark Equivalent Code: ""
--------------------------------------------------------
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col

# 1. Read Silver
df_user = spark.table("spotifyde_catalog.silver.user")

# 2. Create window
window = Window.orderBy("user_id")

# 3. Create surrogate key
df_dim_user = (
    df_user
    .withColumn(
        "DimUserKey",
        row_number().over(window)
    )
    .select(
        "DimUserKey",
        "user_id",
        "user_name",
        "user_country",
        "user_popularity",
        "user_followers",
        "user_following",
        "user_tracks",
        "user_albums",
        "user_artists"
    )
)

# 4. Create Gold Delta table
df_dim_user.write \
    .format("delta") \
    .mode("ignore") \
    .saveAsTable("spotifyde_catalog.gold.dim_user")
    
"""

' Pyspark Equivalent Code: ""\n--------------------------------------------------------\nfrom pyspark.sql.window import Window\nfrom pyspark.sql.functions import row_number, col\n\n# 1. Read Silver\ndf_user = spark.table("spotifyde_catalog.silver.user")\n\n# 2. Create window\nwindow = Window.orderBy("user_id")\n\n# 3. Create surrogate key\ndf_dim_user = (\n    df_user\n    .withColumn(\n        "DimUserKey",\n        row_number().over(window)\n    )\n    .select(\n        "DimUserKey",\n        "user_id",\n        "user_name",\n        "user_country",\n        "user_popularity",\n        "user_followers",\n        "user_following",\n        "user_tracks",\n        "user_albums",\n        "user_artists"\n    )\n)\n\n# 4. Create Gold Delta table\ndf_dim_user.write     .format("delta")     .mode("ignore")     .saveAsTable("spotifyde_catalog.gold.dim_user")\n    \n'

In [0]:
%sql
SELECT *
FROM spotifyde_catalog.gold.dim_user;

/*
                    GOLD
                     │
              ┌──────▼──────┐
              │  dim_user   │
              └─────────────┘
                     │
          ┌──────────┴──────────┐
          │                     │
     user_id               DimUserKey
   Business Key           Surrogate Key

For example:
------------

DimUserKey | user_id | user_name
-----------|---------|-----------
1          | 101     | Carlos
2          | 102     | John

user_id came from the source. 
(i.e. user_id in Gold comes from Silver, which ultimately came from the source SQL database.)

DimUserKey was generated by your warehouse/lakehouse.

*/

DimUserKey,user_id,user_name,country,subscription_type,start_date,end_date,updated_at
1,1,Carlos Berry,Switzerland,Premium,2023-10-17,null,2025-09-23T19:49:55.000Z
2,2,Amanda Jenkins,Montserrat,Family,2024-09-28,null,2025-09-29T19:49:55.000Z
3,3,Daniel Cook,Chile,Premium,2025-07-07,null,2025-09-08T19:49:55.000Z
4,4,Peter Hernandez,Nigeria,Free,2024-07-16,null,2025-09-29T19:49:55.000Z
5,5,Yolanda Morris,Aruba,Premium,2025-05-12,null,2025-09-17T19:49:55.000Z
6,6,Stephen Murphy,Cook Islands,Free,2025-06-17,null,2025-09-12T19:49:55.000Z
7,7,Anthony Andrews Jr.,Libyan Arab Jamahiriya,Free,2024-06-08,null,2025-09-20T19:49:55.000Z
8,8,Felicia Jones,Haiti,Family,2024-09-20,null,2025-09-08T19:49:55.000Z
9,9,Hector Decker,Palau,Family,2024-01-08,null,2025-09-29T19:49:55.000Z
10,10,Frank Davis,New Zealand,Family,2023-12-13,null,2025-09-22T19:49:55.000Z


## SCD1 TYPE

In [0]:
"""
Incoming source data = Silver
Existing target data = Gold

We compare the incoming Silver data with the existing Gold data to determine what has changed.

Silver (Incoming / Source)
          ↓
       COMPARE
          ↓
Gold (Existing / Target)

_______________________________________
                 SCD1
_______________________________________
Silver (Incoming Source)
          ↓
      Compare with
          ↓
Gold (Existing Target)
          ↓
   Change detected?
          ↓
   UPDATE Gold row

Purpose: Detect changes so the existing Gold record can be overwritten.

_______________________________________
           SCD2
_______________________________________
Silver (Incoming Source)
          ↓
      Compare with
          ↓
Gold (Existing Target)
          ↓
   Change detected?
          ↓
 ┌─────────────────────┐
 │ Close old Gold row  │
 │         +           │
 │ Insert new version  │
 └─────────────────────┘

Purpose: Detect changes so the old Gold record can be preserved as history and a new version can be created.

_______________________________________
       In MERGE terminology
_______________________________________
MERGE INTO Gold AS target
USING Silver AS source
ON target.user_id = source.user_id

Silver = incoming source
Gold = existing target

source = alias for Silver
target = alias for Gold

"""

'\nIncoming source data = Silver\nExisting target data = Gold\n\nWe compare the incoming Silver data with the existing Gold data to determine what has changed.\n\nSilver (Incoming / Source)\n          ↓\n       COMPARE\n          ↓\nGold (Existing / Target)\n\n_______________________________________\n                 SCD1\n_______________________________________\nSilver (Incoming Source)\n          ↓\n      Compare with\n          ↓\nGold (Existing Target)\n          ↓\n   Change detected?\n          ↓\n   UPDATE Gold row\n\nPurpose: Detect changes so the existing Gold record can be overwritten.\n\n_______________________________________\n           SCD2\n_______________________________________\nSilver (Incoming Source)\n          ↓\n      Compare with\n          ↓\nGold (Existing Target)\n          ↓\n   Change detected?\n          ↓\n ┌─────────────────────┐\n │ Close old Gold row  │\n │         +           │\n │ Insert new version  │\n └─────────────────────┘\n\nPurpose: Detect chan

In [0]:
%sql

-- IMPORTANT:
--   Run STEP 1 only once.
--   STEP 2 and STEP 3 are for subsequent processing.


-- ============================================================
-- STEP 2: UPDATE EXISTING USERS
-- ============================================================

MERGE INTO spotifyde_catalog.gold.dim_user AS target
USING spotifyde_catalog.silver.user AS source
ON target.user_id = source.user_id

WHEN MATCHED  --user_id matched, but other columns have changed
AND (
    target.user_name             <> source.user_name
    OR target.country            <> source.country
    OR target.subscription_type  <> source.subscription_type
    OR target.start_date         <> source.start_date
    OR target.end_date           <> source.end_date
    OR target.updated_at         <> source.updated_at
)
THEN UPDATE SET 
    target.user_name = source.user_name,
    target.country = source.country,
    target.subscription_type = source.subscription_type,
    target.start_date = source.start_date,
    target.end_date = source.end_date,
    target.updated_at = source.updated_at

--new user
--WHEN NOT MATCHED THEN INSERT *

WHEN NOT MATCHED --safer version, as has DimUserKey also.
THEN INSERT (
    user_id,
    user_name,
    country,
    subscription_type,
    start_date,
    end_date,
    updated_at
)
VALUES (
    source.user_id,
    source.user_name,
    source.country,
    source.subscription_type,
    source.start_date,
    source.end_date,
    source.updated_at
);

-- note: showing modified due to below manual test (bottom of this section)


num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
1,1,0,0


In [0]:
%sql
-- ============================================================
-- STEP 3 : GET INCREMENTAL RECORDS (new users)
-- ============================================================
-- We use updated_at as the watermark.
--
-- MAX(updated_at) in Gold = latest record already processed.
--
-- Therefore, only Silver records newer than that timestamp
-- are sent to the SCD1 logic.
--
-- This prevents us from processing the entire Silver table
-- every time.


CREATE OR REPLACE TEMP VIEW new_users AS

SELECT
    source.user_id,
    source.user_name,
    source.country,
    source.subscription_type,
    source.start_date,
    source.end_date,
    source.updated_at

FROM spotifyde_catalog.silver.user AS source

LEFT JOIN spotifyde_catalog.gold.dim_user AS target
  ON source.user_id = target.user_id

WHERE target.user_id IS NULL;


In [0]:
%sql
-- ============================================================
-- STEP 4: GENERATE SURROGATE KEYS FOR NEW USERS
-- ============================================================
--
-- Existing Gold keys might be:
--
--     1, 2, 3, 4, 5
--
-- MAX(DimUserKey) = 5
--
-- If two new users arrive:
--
--     ROW_NUMBER() = 1, 2
--
-- New keys become:
--
--     5 + 1 = 6
--     5 + 2 = 7
--
-- Therefore, existing surrogate keys remain unchanged.

CREATE OR REPLACE TEMP VIEW new_users_with_keys AS

SELECT

    ROW_NUMBER() OVER (ORDER BY user_id)
    +
    (
        SELECT COALESCE(MAX(DimUserKey), 0)
        FROM spotifyde_catalog.gold.dim_user
    ) AS DimUserKey,

    user_id,
    user_name,
    country,
    subscription_type,
    start_date,
    end_date,
    updated_at

FROM new_users;

In [0]:

%sql
-- ============================================================
-- STEP 5: INSERT NEW USERS INTO GOLD
-- ============================================================
--
-- Only the newly identified users are inserted.
--
-- Existing users are NOT inserted again.
-- ============================================================

INSERT INTO spotifyde_catalog.gold.dim_user
(
    DimUserKey,
    user_id,
    user_name,
    country,
    subscription_type,
    start_date,
    end_date,
    updated_at
)

SELECT

    DimUserKey,
    user_id,
    user_name,
    country,
    subscription_type,
    start_date,
    end_date,
    updated_at

FROM new_users_with_keys;


num_affected_rows,num_inserted_rows
0,0


In [0]:
%sql
-- ============================================================
-- FINAL RESULT
-- ============================================================
--
-- SCD TYPE 1:
--
-- Existing user:
--     UPDATE attributes
--     KEEP same DimUserKey
--
-- New user:
--     GENERATE new DimUserKey
--     INSERT new record
--
-- Old values are overwritten.
-- Historical versions are NOT maintained.

In [0]:
%sql
DESCRIBE DETAIL spotifyde_catalog.gold.dim_user;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,40f23f07-c1e5-4a1e-846a-b1e34774da1e,spotifyde_catalog.gold.dim_user,null,abfss://gold@adlspotifyde.dfs.core.windows.net/dim_user,2026-09-15T10:17:48.542Z,2026-09-15T10:24:40.000Z,List(),List(),1,17351,Map(delta.enableDeletionVectors -> true),3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%sql
SELECT
    COUNT(*) AS total_rows
FROM spotifyde_catalog.gold.dim_user;


total_rows
500


## Test SCD1 with sample as below
*assuming <If you don't have an actual source DB available, temporarily change Silver>

Else ideal way:
```text
Source DB
   │
   │ Business change
   │ country: Switzerland → India (for user_id=1)
   ▼
ADF / CDC / Incremental Load
   │
   ▼
Bronze
   │
   ▼
Silver
   │
   │ Cleaned data
   ▼
Gold — SCD1 MERGE
   │
   ├── Existing user → UPDATE attributes
   │                   Keep same DimUserKey
   │
   └── New user      → INSERT
                       Generate new DimUse
```


In [0]:
%sql
--Original data
SELECT * FROM spotifyde_catalog.gold.dim_user 
WHERE user_id = 1

DimUserKey,user_id,user_name,country,subscription_type,start_date,end_date,updated_at
1,1,Carlos Berry,Switzerland,Premium,2023-10-17,null,2025-09-23T19:49:55.000Z


In [0]:
%sql
-- Modified temporarily user_id=1 country
UPDATE spotifyde_catalog.silver.user
SET country = 'India',
    updated_at = current_timestamp()
WHERE user_id = 1;

--then RUN
-- STEP 2: UPDATE EXISTING USERS

num_affected_rows
1


In [0]:
%sql
-- Check the result
SELECT *
FROM spotifyde_catalog.gold.dim_user
WHERE user_id = 1;

DimUserKey,user_id,user_name,country,subscription_type,start_date,end_date,updated_at
1,1,Carlos Berry,India,Premium,2023-10-17,null,2026-09-15T10:35:52.412Z



## Revert Modified Change
revert the change using Delta Lake's time travel / restore features.

In [0]:
%sql
DESCRIBE HISTORY spotifyde_catalog.silver.user;

--both tables were changed:
--Silver was changed by your manual UPDATE
--Gold was changed by the MERGE

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
2,2026-09-15T10:35:57.000Z,147247804551352,nitinkumar21038_gmail.com#ext#@nitinkumar21038gmail.onmicrosoft.com,UPDATE,"Map(predicate -> [""(user_id#15619 = 1)""])",null,List(1437253803066914),fa0ecac6-34f7-4e86-8c38-66d6306c6c95,0915-095337-rza3agh2-v2n,1,WriteSerializable,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 2598, numCopiedRows -> 1, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 3497, numDeletionVectorsUpdated -> 0, scanTimeMs -> 562, numAddedFiles -> 1, numUpdatedRows -> 1, numAddedBytes -> 2613, rewriteTimeMs -> 2929)",null,Databricks-Runtime/19.6.x-aarch64-photon-scala2.13
1,2026-09-15T09:54:10.000Z,147247804551352,nitinkumar21038_gmail.com#ext#@nitinkumar21038gmail.onmicrosoft.com,STREAMING UPDATE,"Map(outputMode -> Append, queryId -> 73b3f80b-9831-4a64-b1dc-8b6155cd4426, epochId -> 0, statsOnLoad -> false)",null,List(1816100660469171),b9ebe9d0-8812-4c51-b8ee-9ebb4ab10e8f,0915-091502-63qjdmpr-v2n,0,WriteSerializable,true,"Map(numRemovedFiles -> 0, numOutputRows -> 500, numOutputBytes -> 492638, numAddedFiles -> 184)",null,Databricks-Runtime/19.6.x-aarch64-photon-scala2.13
0,2026-09-15T09:53:08.000Z,147247804551352,nitinkumar21038_gmail.com#ext#@nitinkumar21038gmail.onmicrosoft.com,STREAMING UPDATE,"Map(outputMode -> Append, queryId -> 73b3f80b-9831-4a64-b1dc-8b6155cd4426, epochId -> -1, statsOnLoad -> false)",null,List(1816100660469171),b9ebe9d0-8812-4c51-b8ee-9ebb4ab10e8f,0915-091502-63qjdmpr-v2n,null,WriteSerializable,true,Map(),null,Databricks-Runtime/19.6.x-aarch64-photon-scala2.13


In [0]:
%sql
SELECT *
FROM spotifyde_catalog.silver.user VERSION AS OF 2
WHERE user_id = 1;

--and

user_id,user_name,country,subscription_type,start_date,end_date,updated_at,first_name,last_name
1,Carlos Berry,India,Premium,2023-10-17,null,2026-09-15T10:35:52.412Z,Carlos,Berry


In [0]:
%sql

SELECT *
FROM spotifyde_catalog.silver.user VERSION AS OF 1
WHERE user_id = 1;

user_id,user_name,country,subscription_type,start_date,end_date,updated_at,first_name,last_name
1,Carlos Berry,Switzerland,Premium,2023-10-17,null,2025-09-23T19:49:55.000Z,Carlos,Berry


In [0]:
%sql

RESTORE TABLE spotifyde_catalog.silver.user
TO VERSION AS OF 1;

table_size_after_restore,num_of_files_after_restore,num_removed_files,num_restored_files,removed_files_size,restored_files_size
492638,184,1,1,2613,2598



Changing an existing user in Silver is enough.

When the Gold SCD1 pipeline runs again, the MERGE reads the latest Silver data
and updates the matching Gold record while preserving the existing `DimUserKey`.

In [0]:
%sql
-- Check the result
-- after RUN
-- STEP 2: UPDATE EXISTING USERS
SELECT *
FROM spotifyde_catalog.gold.dim_user
WHERE user_id = 1;

DimUserKey,user_id,user_name,country,subscription_type,start_date,end_date,updated_at
1,1,Carlos Berry,Switzerland,Premium,2023-10-17,null,2025-09-23T19:49:55.000Z
